# Imports

In [8]:
from sklearn.datasets import make_classification
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold

## Utils

In [3]:
def generate_synthetic_binary_data(
    n_samples=10000,
    n_features=30,
    n_informative=5,
    n_redundant=10,
    n_repeated=0,
    n_classes=2,
    class_sep=0.5,
    flip_y=0.15,
    weights=[0.8, 0.2],
    test_size=0.2,
    valid_size=0.2,
    random_state=42
):
    """
    Gera uma base sintética binária com splits train/valid/test.
    Retorna: (X_train, y_train, X_valid, y_valid, X_test, y_test)
    """

    # ---------------------------
    # 1. Gerar base sintética
    # ---------------------------
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_repeated=n_repeated,
        n_classes=n_classes,
        class_sep=class_sep,
        flip_y=flip_y,
        weights=weights,
        random_state=random_state
    )

    # ---------------------------
    # 2. Train/Test
    # ---------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # ---------------------------
    # 3. Train/Validation
    # ---------------------------
    valid_relative = valid_size / (1 - test_size)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train, y_train,
        test_size=valid_relative,
        stratify=y_train,
        random_state=random_state
    )

    y_train = pd.DataFrame(y_train, columns=['target'])
    y_valid = pd.DataFrame(y_valid, columns=['target'])
    y_test = pd.DataFrame(y_test, columns=['target'])
    
    return pd.DataFrame(X_train), y_train, pd.DataFrame(X_valid), y_valid, pd.DataFrame(X_test), y_test

# Tests

In [4]:
X_train, y_train, X_valid, y_valid, X_test, y_test = generate_synthetic_binary_data(
    n_samples=5000,
    n_features=30,
    n_informative=5,
    n_redundant=10,
    n_repeated=0,
    n_classes=2,
    class_sep=0.5,   
    flip_y=0.15,     
    weights=[0.8, 0.2], 
    test_size=0.2,
    valid_size=0.2,
    random_state=42
)

In [16]:
from __future__ import annotations

from typing import Iterable, Sequence
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class FeatureSelector(BaseEstimator, TransformerMixin):
    """Select a subset of features from a DataFrame or NumPy array.

    Parameters
    ----------
    features : Iterable[str] or Iterable[int], default=None
        List of column names (for pandas) or column indices (for numpy/pandas) 
        to select.

    mask : Iterable[bool], default=None
        Boolean mask of length n_features_in_ indicating which features to select.

    check_missing : bool, default=True
        If True, validates whether the requested features exist in the input data.
        Only applicable when `features` are provided as strings.

    Attributes
    ----------
    selected_features_ : list
        List of selected feature names (strings) or indices (ints).

    n_features_in_ : int
        Number of features seen during `fit`.

    feature_names_in_ : ndarray of shape (n_features_in_,)
        Names of features seen during `fit`. Defined only when `X` has feature
        names (e.g., a pandas DataFrame) or generated as string indices for numpy.
    """

    def __init__(
        self, 
        features: Iterable[str | int] | None = None, 
        mask: Iterable[bool] | None = None, 
        check_missing: bool = True
    ):
        self.features = features
        self.mask = mask
        self.check_missing = check_missing

    def fit(self, X, y=None):
        """Fit the feature selector to the input data.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            The training input samples.
        y : None
            Ignored. This parameter exists only for compatibility with 
            sklearn's Pipeline.

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        
        if self.features is None and self.mask is None:
            raise ValueError("Either 'features' or 'mask' must be provided.")
        if self.features is not None and self.mask is not None:
            raise ValueError("Only one of 'features' or 'mask' should be provided.")

        
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
            self.n_features_in_ = X.shape[1]
            is_pandas = True

        else:
            X_arr = np.asarray(X)
            self.n_features_in_ = X_arr.shape[1]
            self.feature_names_in_ = np.array([f"x{i}" for i in range(self.n_features_in_)], dtype=object) # Generate synthetic names for numpy alignment
            is_pandas = False

        # Case 1: Mask-based selection
        if self.mask is not None:
            mask = np.asarray(self.mask, dtype=bool)
            if len(mask) != self.n_features_in_:
                raise ValueError(
                    f"Mask length ({len(mask)}) does not match "
                    f"number of features ({self.n_features_in_})."
                )
            
            # Save selection based on input type
            if is_pandas:
                self.selected_features_ = list(self.feature_names_in_[mask])
            else:
                self.selected_features_ = list(np.where(mask)[0])

        # Case 2: Explicit features list (names or indices)
        else:
            features_list = list(self.features)
            
            # If strings are passed but input is numpy, try to map from synthetic names or indices
            if not is_pandas and all(isinstance(f, str) for f in features_list):
                # If they passed synthetic names like ['x0', 'x2']
                if all(f in self.feature_names_in_ for f in features_list):
                    self.selected_features_ = [list(self.feature_names_in_).index(f) for f in features_list]
                else:
                    raise ValueError("String feature names cannot be mapped to a NumPy array unless they match 'x0', 'x1', etc.")
            else:
                self.selected_features_ = features_list

            # Optional check for missing columns (only makes sense for string names in pandas)
            if self.check_missing and is_pandas and all(isinstance(f, str) for f in self.selected_features_):
                missing = sorted(set(self.selected_features_) - set(self.feature_names_in_))
                if missing:
                    raise ValueError(f"The following features do not exist in X: {missing}")

        return self

    def transform(self, X):
        """Reduce X to the selected features.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_features)
            The input samples.

        Returns
        -------
        X_sliced : {array-like, sparse matrix} of shape (n_samples, n_selected_features)
            The input samples with only the selected features.
        """
        check_is_fitted(self, attributes=["selected_features_", "n_features_in_"])

        if isinstance(X, pd.DataFrame):
            # If fit was on pandas or indices are used, pandas .loc/.iloc handles it safely
            if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
                return X.iloc[:, self.selected_features_]
            return X.loc[:, self.selected_features_]
        
        else:
            X_arr = np.asarray(X)
            # If selected_features_ contains string names (from pandas fit) but X is numpy
            if Janus_indices := [isinstance(f, str) for f in self.selected_features_]:
                if any(Janus_indices):
                    # Map string names back to positions using stored feature_names_in_
                    indices = [list(self.feature_names_in_).index(f) for f in self.selected_features_]
                    return X_arr[:, indices]
            
            return X_arr[:, self.selected_features_]

    def inverse_transform(self, X):
        """Reverse the transformation, filling unselected features with NaNs.

        Parameters
        ----------
        X : {array-like, sparse matrix} of shape (n_samples, n_selected_features)
            The converted input samples.

        Returns
        -------
        X_original : {array-like, sparse matrix} of shape (n_samples, n_features_in_)
            The original structure filled with NaNs where features were excluded.
        """
        check_is_fitted(self, attributes=["selected_features_", "feature_names_in_"])

        if isinstance(X, pd.DataFrame):
            out = pd.DataFrame(index=X.index, columns=self.feature_names_in_)
            if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
                cols = self.feature_names_in_[self.selected_features_]
                out[cols] = X.values
            else:
                out[self.selected_features_] = X
            return out
        
        else:
            X_arr = np.asarray(X)
            out = np.full((X_arr.shape[0], self.n_features_in_), np.nan, dtype=float)
            
            # Map features to numeric indices for array assignment
            if all(isinstance(f, str) for f in self.selected_features_):
                indices = [list(self.feature_names_in_).index(f) for f in self.selected_features_]
            else:
                indices = self.selected_features_
                
            out[:, indices] = X_arr
            return out

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        """Get output feature names for transformation."""
        check_is_fitted(self, attributes=["selected_features_"])
        if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
            return np.asarray(self.feature_names_in_[self.selected_features_], dtype=object)
        return np.asarray(self.selected_features_, dtype=object)

    def get_support(self, indices: bool = False) -> np.ndarray:
        """Get a mask or index array of the features selected."""
        check_is_fitted(self, attributes=["selected_features_", "feature_names_in_"])
        
        if all(isinstance(f, (int, np.integer)) for f in self.selected_features_):
            mask = np.zeros(self.n_features_in_, dtype=bool)
            mask[self.selected_features_] = True
        else:
            mask = np.isin(self.feature_names_in_, self.selected_features_)

        if indices:
            return np.where(mask)[0]
        return mask

    @property
    def features_(self) -> list:
        """Backward compatibility helper for selected features."""
        check_is_fitted(self, attributes=["selected_features_"])
        return self.selected_features_

    def __len__(self) -> int:
        check_is_fitted(self, attributes=["selected_features_"])
        return len(self.selected_features_)

    def __repr__(self) -> str:
        try:
            length = len(self)
        except Exception:
            length = "Not Fitted"
        return f"FeatureSelector(n_features={length})"

In [17]:
X_train.head()

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.660424,0.370553,-0.157266,0.763497,0.291066,0.031432,0.269237,1.068864,0.106731,2.140277,...,-2.033320,0.228399,-0.886704,-0.176886,-0.213807,0.250023,-0.130981,2.060301,0.615700,1.099804
1,-1.332327,-1.531204,-0.768285,0.823467,-0.504803,-0.891358,-1.173892,0.610270,-0.127102,-0.738951,...,2.745836,1.374807,-1.812954,-1.929477,1.686287,1.374261,0.599295,0.961674,-0.668937,-0.009860
2,-0.411217,-1.182707,1.488460,0.452974,1.215235,1.832060,0.139225,0.171208,0.413916,0.530517,...,-1.074884,0.741791,-1.394438,-1.329335,0.439295,-1.511498,1.691682,1.035581,1.252118,-1.339890
3,-0.961314,2.793710,3.721175,-1.254595,2.116411,4.164159,3.264257,-0.149023,1.007158,0.991212,...,-0.036316,-1.537561,-1.527298,4.315490,-0.549563,-0.684245,-0.067385,-0.730745,2.883616,1.313439
4,0.374048,-0.036991,-0.108653,-0.128614,-0.039760,0.146764,-0.122993,-1.256825,0.297515,0.820744,...,-0.514466,-0.215456,-1.477026,-0.440279,-0.959748,0.786763,-0.244742,0.429517,1.167471,1.183787


In [20]:
model = make_pipeline(
    FeatureSelector(features=[0, 1, 2, 3, 4, 20, 21]),
    LogisticRegression()
)

model.fit(X_train, y_train)

/home/junior/Documentos/GitHub/data-science-toolkit/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Pipeline(steps=[('featureselector',
                 FeatureSelector(features=[0, 1, 2, 3, 4, 20, 21])),
                ('logisticregression', LogisticRegression())])

In [21]:
model.named_steps['logisticregression'].coef_

array([[-0.01555216, -0.14602427, -0.18812237, -0.09062114,  0.51275496,
        -0.05528237,  0.08349947]])